# Biohub Cell Tracking — Kaggle Submission Notebook

Runs inference on the hidden test set and writes `submission.csv`.

**Inputs expected:**
- `/kaggle/input/biohub-cell-tracking-during-development/test/` — test .zarr volumes
- `/kaggle/input/<your-model-dataset>/best.pt` — trained checkpoint (automatically resolved)
- `/kaggle/input/<your-model-dataset>/src/` — source package (automatically resolved)

**Output:** `/kaggle/working/submission.csv`

In [1]:
# ── Install / upgrade dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('zarr>=3.0', 'scipy>=1.10', 'pandas>=2.0', 'tqdm', 'blosc2', 'scikit-image>=0.21')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 100.2 MB/s eta 0:00:00


In [2]:
# ── Paths & Environment Setup ───────────────────────────────────────────────────
import sys
from pathlib import Path

OUTPUT_CSV    = Path('/kaggle/working/submission.csv')

# Ensure /kaggle/working is in path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Fast path resolution: Scan immediate subdirectories of /kaggle/input and skip the competition dataset
input_root = Path("/kaggle/input")
CKPT_PATH = "/kaggle/input/datasets/freak2209/submision-weights/src/swa_final.pt"
TEST_ROOT = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"

if input_root.exists():
    search_paths = [input_root]
    competitions_dir = input_root / "competitions"
    if competitions_dir.exists():
        search_paths.append(competitions_dir)
        
    for parent in search_paths:
        for child in parent.iterdir():
            if "biohub-cell-tracking-during-development" in child.name:
                # Search for test Zarrs
                candidates = []
                for pattern in ["*.zarr", "*/*.zarr", "*/*/*.zarr"]:
                    candidates.extend(list(child.glob(pattern)))
                if candidates:
                    test_candidates = [p for p in candidates if "test" in p.parts or "test" in str(p)]
                    if test_candidates:
                        TEST_ROOT = test_candidates[0].parent
                    else:
                        TEST_ROOT = candidates[0].parent
                continue  # Bypassed the massive competition directory recursively
                
            if child.is_dir():
                # Search for src
                if child.name == "src":
                    if str(child.parent) not in sys.path:
                        sys.path.insert(0, str(child.parent))
                        print(f"Added source directory to sys.path: {child.parent}")
                else:
                    for p in child.glob("src"):
                        if p.is_dir() and str(child) not in sys.path:
                            sys.path.insert(0, str(child))
                            print(f"Added source directory to sys.path: {child}")
                

if TEST_ROOT is None:
    TEST_ROOT = Path('/kaggle/input/biohub-cell-tracking-during-development/test')

if CKPT_PATH is None:
    # Fallback to local working directory
    CKPT_PATH = Path('/kaggle/working/best.pt')

print(f'Test root  : {TEST_ROOT}  (exists={TEST_ROOT.exists()})')
print(f'Checkpoint : {CKPT_PATH}  (exists={CKPT_PATH.exists()})')

Test root  : /kaggle/input/competitions/biohub-cell-tracking-during-development/test  (exists=True)
Checkpoint : /kaggle/input/datasets/freak2209/submision-weights/src/best.pt  (exists=True)


In [3]:
# ── Build config programmatically ───────────────────────────────────────────────
import sys
sys.path.insert(0, "/kaggle/input/datasets/freak2209/biohub-cell-tracking-src")
from src.config import Config, ModelConfig, DataConfig, TrainingConfig, InferenceConfig

config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=16, # Fixed to match checkpoint!
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=3,      # Match training
        nms_threshold=0.5, # Match training
    ),
    data=DataConfig(
        data_root=str(TEST_ROOT),
        input_shape=[64, 256, 256],
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode="percentile",
    ),
    inference=InferenceConfig(
        checkpoint=str(CKPT_PATH),
        max_link_distance_um=7.0,
        output=str(OUTPUT_CSV),
    ),
)
print("Config ready.")


Config ready.


In [4]:
# ── Load model ───────────────────────────────────────────────────────────────────
import torch
from src.models.residual_unet3d import ResidualCellTrackingUNet
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = ResidualCellTrackingUNet.from_config(config)
if Path(CKPT_PATH).exists():
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt.get('model', ckpt))
    print(f'Loaded checkpoint from {CKPT_PATH}')
else:
    print(f'WARNING: Checkpoint {CKPT_PATH} not found!')
model.to(device)
model.eval()
print(f'Model output shape: {model.output_shape}')


Device: cuda
Loaded checkpoint from /kaggle/input/datasets/freak2209/submision-weights/src/best.pt
Model output shape: (16, 164, 164)


In [ ]:
# ─── Run inference ─────────────────────────────────────────────────────────────────────────────
from src.infer import tracking_result_to_submission_rows, compute_crop_offset
from src.tracking.linker import extract_peaks_from_frame
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
import json
import os
import blosc2
import glob

def pad_spatial_tensor_if_needed(x, min_shape=(64, 256, 256)):
    spatial_shape = x.shape[-3:]
    pad_z = max(0, min_shape[0] - spatial_shape[0])
    pad_y = max(0, min_shape[1] - spatial_shape[1])
    pad_x = max(0, min_shape[2] - spatial_shape[2])
    if pad_z == 0 and pad_y == 0 and pad_x == 0:
        return x, (0, 0, 0)
    pz_b, pz_a = pad_z // 2, pad_z - pad_z // 2
    py_b, py_a = pad_y // 2, pad_y - pad_y // 2
    px_b, px_a = pad_x // 2, pad_x - pad_x // 2
    pad_tuple = (px_b, px_a, py_b, py_a, pz_b, pz_a)
    if x.dim() == 6:
        N, C, T, Z, Y, X = x.shape
        x_5d = x.view(N, C * T, Z, Y, X)
        padded_5d = torch.nn.functional.pad(x_5d, pad_tuple, mode="reflect")
        padded_x = padded_5d.view(N, C, T, padded_5d.shape[2], padded_5d.shape[3], padded_5d.shape[4])
    else:
        padded_x = torch.nn.functional.pad(x, pad_tuple, mode="reflect")
    return padded_x, (pz_b, py_b, px_b)

def normalize_volume(volume, mode="percentile", percentile=99.5):
    vol = volume.astype(np.float32)
    if mode == "per_volume":
        vmax = float(vol.max())
        if vmax > 0:
            vol /= vmax
    elif mode == "percentile":
        vmax = float(np.percentile(vol, percentile))
        if vmax > 0:
            vol = np.clip(vol / vmax, 0.0, 1.0)
    return vol

samples = sorted(glob.glob(str(TEST_ROOT) + "/*.zarr"))
scale = tuple(config.data.scale)
min_shape = tuple(config.data.input_shape)
all_rows = []
row_id = 0

for sample_path in tqdm(samples, desc="Samples"):
    sample_id = os.path.basename(sample_path).replace(".zarr", "")
    with open(os.path.join(sample_path, "0", "zarr.json")) as f:
        meta = json.load(f)
        shape = meta["shape"]
        chunk_shape = meta["chunk_grid"]["chunk_shape"] if "chunk_shape" in meta["chunk_grid"] else meta["chunk_grid"]["configuration"]["chunk_shape"]
    
    num_timepoints = shape[0]
    input_shape = tuple(shape[1:])
    frame_detections = {}
    
    def get_frame(t):
        chunk_path = os.path.join(sample_path, "0", "c", str(t), "0", "0", "0")
        with open(chunk_path, "rb") as f:
            decomp = blosc2.decompress(f.read())
        return np.frombuffer(decomp, dtype=np.uint16).reshape(chunk_shape).squeeze(0)
    
    for t in tqdm(range(num_timepoints), desc=sample_id, leave=False):
        frames = []
        for i in range(-3, 4):
            t_offset = max(0, min(t + i, num_timepoints - 1))
            f = normalize_volume(get_frame(t_offset), mode=config.data.normalize_mode)
            frames.append(f)
        frame_window = np.stack(frames, axis=0)
        x = torch.from_numpy(frame_window).unsqueeze(0).unsqueeze(0).float().to(device)
        x_padded, (pz_b, py_b, px_b) = pad_spatial_tensor_if_needed(x, min_shape=min_shape)
        
        with torch.no_grad():
            outputs = model(x_padded, inference=True)
            ci     = outputs["cell_indicator"][0, 0].cpu().numpy()
            maxima = outputs["maxima"][0].cpu().numpy() if outputs["maxima"].ndim == 4 else outputs["maxima"][0, 0].cpu().numpy()
            mv     = outputs["movement_vectors"][0].cpu().numpy()
        
        actual_output_shape = ci.shape
        crop_offset_padded = compute_crop_offset(x_padded.shape[-3:], actual_output_shape)
        crop_offset = (
            crop_offset_padded[0] - pz_b,
            crop_offset_padded[1] - py_b,
            crop_offset_padded[2] - px_b,
        )
        nodes = extract_peaks_from_frame(ci, maxima, mv, t=t, threshold=config.model.nms_threshold, crop_offset=crop_offset)
        frame_detections[t] = nodes
        
    rows, row_id = tracking_result_to_submission_rows(
        sample_id=sample_id,
        frame_detections=frame_detections,
        max_distance_um=config.inference.max_link_distance_um,
        scale=scale,
        allow_divisions=True,
        start_id=row_id,
    )
    all_rows.extend(rows)

columns = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
df = pd.DataFrame(all_rows)
if df.empty: df = pd.DataFrame(columns=columns)
else: df = df[columns]

print(f"Total rows: {len(df):,}   (nodes + edges)")
df.head(10)


Loaded checkpoint from /kaggle/input/datasets/freak2209/submision-weights/src/best.pt


44b6_0b24845f:   8%|▊         | 8/100 [00:14<02:37,  1.71s/it]

In [ ]:
# ── Validate submission format ───────────────────────────────────────────────────
REQUIRED_COLS = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
assert list(df.columns) == REQUIRED_COLS, f'Column mismatch: {list(df.columns)}'

node_rows = df[df.row_type == 'node']
edge_rows = df[df.row_type == 'edge']
print(f'Nodes : {len(node_rows):,}')
print(f'Edges : {len(edge_rows):,}')
print(f'Datasets in submission: {df.dataset.nunique()}')

# Node rows should have valid (non-negative) coordinates
assert (node_rows[['t','z','y','x']] >= 0).all().all(), 'Negative node coordinates!'
# Edge rows must have valid source/target IDs
assert (edge_rows[['source_id','target_id']] > 0).all().all(), 'Invalid edge IDs!'

print('All checks passed.')

In [ ]:
# ── Save submission ───────────────────────────────────────────────────────────────
df.to_csv(str(OUTPUT_CSV), index=False)
print(f'submission.csv written → {OUTPUT_CSV}  ({OUTPUT_CSV.stat().st_size / 1024:.1f} KB)')